<a href="https://colab.research.google.com/github/ryouchinsa/Rectlabel-support/blob/master/notebooks/train_rf_detr160_instance_segmentation_on_custom_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How to Train a RF-DETR Instance Segmentation Model with Custom Data

We will show you how to train a RF-DETR instance segmentation model with your images and annotations and export to a Core ML model which can be used for auto labeling on RectLabel.

### Use GPU

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Runtime` -> `Change runtime type` -> `Hardware accelerator`, set it to `GPU`, and then click `Save`.

In [ ]:
!nvidia-smi

### Install PyTorch 2.8.0

In [ ]:
!pip install -q torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128


### Install RF-DETR 1.6.0

In [ ]:
!pip install -q rfdetr[train,loggers]==1.6.0

### Download training images and annotations

Download training images and annotations. You can use these or replace them with your own data.

In [ ]:
!mkdir datasets
%cd datasets
!wget -q https://huggingface.co/datasets/rectlabel/datasets/resolve/main/donut_coco.zip
!unzip -q donut_coco.zip
%cd ..

### Fine-tune RF-DETR on custom dataset

Start training from the current content folder.

In [ ]:
from rfdetr import RFDETRSegNano

model = RFDETRSegNano()
dataset_dir = "datasets/donut_coco"
model.train(dataset_dir=dataset_dir, epochs=20, batch_size=4, grad_accum_steps=4)

The trained model is checkpoint_best_total.pth.

In [ ]:
!ls -la /content/output

Before exporting to a Core ML model, edit a line of transformer.py.

In [ ]:
!pip show rfdetr

In [ ]:
path = "/usr/local/lib/python3.12/dist-packages/rfdetr/models/transformer.py"
with open(path, "r") as f:
    content = f.read()
modified_content = content.replace("mask_flatten, spatial_shapes_hw", "mask_flatten, spatial_shapes")
with open(path, "w") as f:
    f.write(modified_content)

### Install RF-DETR to CoreML

In [ ]:
!git clone https://github.com/landchenxuan/rf-detr-to-coreml.git
%cd rf-detr-to-coreml
!pip install -q -e .

Move the best model to the current folder and export to a Core ML model.

In [ ]:
!mv /content/output/checkpoint_best_total.pth .

In [ ]:
!rfdetr-coreml --model seg-nano --weights checkpoint_best_total.pth

In [ ]:
!ls -la output

In [ ]:
%cd output

Zip the Core ML model and download it from the File browser at the left hand. You can auto label images using the Core ML model on RectLabel.

In [ ]:
!zip -r seg-nano.zip rf-detr-seg-nano-checkpoint_best_total-fp32.mlpackage